# Hands-On 9: Learning across datasets

Record a prediction before running each experiment.

In [ ]:
from pathlib import Path
import os
import sys
try:
    import mlcourse.setup
except ModuleNotFoundError:
    bases = [Path(os.environ.get("MLCOURSE_ROOT", Path.cwd())), Path.cwd(), Path("/content/pp-machine-learning")]
    for base in bases:
        for candidate in (base.resolve(), *base.resolve().parents):
            if (candidate / "src/mlcourse/setup.py").is_file():
                sys.path.insert(0, str(candidate / "src"))
                break
        else:
            continue
        break
    else:
        raise RuntimeError("Course files not found. Open the extracted course repository or set MLCOURSE_ROOT to its location.") from None

In [ ]:
from mlcourse.setup import setup_notebook
REPO_ROOT = setup_notebook()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.tree import DecisionTreeRegressor, plot_tree
from mlcourse.labs import load_meta_dataset
from mlcourse.widgets import interactive_meta_performance

## 1. Inspect a different kind of observation

Load the meta-learning dataframe.

**Prediction:** What does one row represent here?

*Your response.*

In [ ]:
meta_data = load_meta_dataset()
display(meta_data.head())
print(f'Datasets: {len(meta_data)}')

**Observation:** Identify the dataset-level descriptors and algorithm-performance columns.

*Your response.*

**Explanation:** How does this differ from predicting the class of an individual observation?

*Your response.*

## 2. Compare model capacity across datasets

Compute full-tree accuracy minus stump accuracy and inspect the ordering.

**Prediction:** Can a simpler stump outperform the full tree on some datasets? Why?

*Your response.*

In [ ]:
ordered = meta_data.sort_values('accuracy_gap')
display(ordered[['dataset', 'tree_accuracy', 'stump_accuracy', 'accuracy_gap']])
fig, ax = plt.subplots(figsize=(9, 5), layout='constrained')
ax.barh(ordered.dataset, ordered.accuracy_gap)
ax.axvline(0, color='black', linewidth=1)
ax.set(xlabel='Full-tree accuracy minus stump accuracy', ylabel='Dataset')
plt.show()

**Observation:** Which datasets favour each tree model?

*Your response.*

**Explanation:** Explain how model capacity can contribute to this variation.

*Your response.*

## 3. Relate task descriptors to the performance gap

Plot class imbalance and feature count against the full-tree minus stump accuracy gap.

**Prediction:** Should either descriptor determine the winner perfectly?

*Your response.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
for ax, feature in zip(axes, ['class_imbalance', 'n_features']):
    ax.scatter(meta_data[feature], meta_data.accuracy_gap, s=55)
    ax.axhline(0, color='black', linewidth=1)
    ax.set(xlabel=feature, ylabel='Accuracy gap', title=f'Gap versus {feature}')
plt.show()

**Observation:** Look for different outcomes at similar descriptor values.

*Your response.*

**Explanation:** Why can one dataset summary conceal important structure in the underlying feature space?

*Your response.*

## 4. Predict algorithm performance

A binary winner discards information. Instead, predict the measured accuracy of three candidate strategies for an unseen dataset: a full decision tree, a decision stump, and a majority-class baseline. Each leave-one-dataset-out prediction is made without using the held-out dataset's measured accuracies.

**Prediction:** What advantage is gained by predicting performance values instead of only predicting which model wins?

*Your response.*

In [ ]:
meta_features = ['n_instances', 'n_features', 'n_classes', 'class_imbalance', 'numeric_ratio', 'class_entropy']
performance_columns = ['tree_accuracy', 'stump_accuracy', 'majority_accuracy']
strategy_names = {
    'tree_accuracy': 'Full tree',
    'stump_accuracy': 'Decision stump',
    'majority_accuracy': 'Majority baseline',
}

X_meta = meta_data[meta_features]
y_performance = meta_data[performance_columns]
loo = LeaveOneOut()
meta_regressor = RandomForestRegressor(
    n_estimators=300,
    max_depth=3,
    min_samples_leaf=2,
    random_state=42,
)
predicted_performance = cross_val_predict(meta_regressor, X_meta, y_performance, cv=loo)

observed = y_performance.to_numpy()
mean_baseline = np.empty_like(observed)
for held_out in range(len(meta_data)):
    train_mask = np.arange(len(meta_data)) != held_out
    mean_baseline[held_out] = observed[train_mask].mean(axis=0)

performance_predictions = pd.DataFrame({'dataset': meta_data.dataset})
for index, column in enumerate(performance_columns):
    performance_predictions[f'pred_{column}'] = predicted_performance[:, index]

meta_mae = mean_absolute_error(observed, predicted_performance)
baseline_mae = mean_absolute_error(observed, mean_baseline)
print(f'Meta-model MAE: {meta_mae:.3f}')
print(f'Mean-performance baseline MAE: {baseline_mae:.3f}')

**Observation:** Compare the meta-model MAE with the mean-performance baseline MAE.

*Your response.*

**Explanation:** What does a lower MAE mean at the meta-level, and what does it not prove?

*Your response.*

## 5. Turn predicted performance into a ranking

For each held-out dataset, rank the candidate strategies by their predicted accuracy and compare the top recommendation with the measured best strategy.

**Prediction:** Can a model improve performance estimates without improving the top recommendation on every dataset?

*Your response.*

In [ ]:
actual_best = observed.argmax(axis=1)
predicted_best = predicted_performance.argmax(axis=1)
baseline_best = mean_baseline.argmax(axis=1)
labels = np.array([strategy_names[column] for column in performance_columns])

ranking_summary = pd.DataFrame({
    'dataset': meta_data.dataset,
    'predicted_best': labels[predicted_best],
    'measured_best': labels[actual_best],
    'correct_recommendation': predicted_best == actual_best,
})
display(ranking_summary)
print(f'Meta-model top recommendation accuracy: {(predicted_best == actual_best).mean():.3f}')
print(f'Mean-performance baseline top recommendation accuracy: {(baseline_best == actual_best).mean():.3f}')

fig, ax = plt.subplots(figsize=(7, 6), layout='constrained')
markers = ['o', 's', '^']
for index, column in enumerate(performance_columns):
    ax.scatter(observed[:, index], predicted_performance[:, index], marker=markers[index], s=55,
               label=strategy_names[column])
limits = [min(observed.min(), predicted_performance.min()) - .03,
          max(observed.max(), predicted_performance.max()) + .03]
ax.plot(limits, limits, color='black', linewidth=1)
ax.set(xlim=limits, ylim=limits, xlabel='Measured accuracy', ylabel='Predicted accuracy',
       title='Leave-one-dataset-out performance prediction')
ax.legend()
plt.show()

**Observation:** Inspect the predicted-versus-measured plot and the recommendation table.

*Your response.*

**Explanation:** Why is predicting approximate performance a richer meta-learning task than predicting only a winner label?

*Your response.*

## 6. Inspect one recommendation interactively

Select a dataset, a meta-feature, and a candidate strategy. Compare predicted and measured performance and inspect the resulting ranking.

**Prediction:** Choose one dataset before interacting. Which strategy do you expect to rank first?

*Your response.*

In [ ]:
performance_lab = interactive_meta_performance(meta_data, performance_predictions)
display(performance_lab.widget)

**Observation:** Find one dataset for which the predicted ranking is close to the measured ranking and one where it is not.

*Your response.*

**Explanation:** Which dataset-level descriptors might help explain the difference?

*Your response.*

## 7. Use a shallow tree as an explanation lens

The random forest above is the evaluated performance predictor. Fit a separate shallow regression tree only to visualise how a few meta-feature thresholds can partition dataset space when explaining full-tree performance.

**Prediction:** What does a leaf value represent in this meta-tree?

*Your response.*

In [ ]:
explanation_tree = DecisionTreeRegressor(max_depth=2, min_samples_leaf=2, random_state=0)
explanation_tree.fit(meta_data[meta_features], meta_data.tree_accuracy)
fig, ax = plt.subplots(figsize=(13, 5), layout='constrained')
plot_tree(explanation_tree, feature_names=meta_features, filled=True, rounded=True, precision=3, ax=ax)
ax.set_title('A shallow explanation tree for full-tree accuracy')
plt.show()

**Observation:** Trace one path from the root to a leaf and state the meta-feature conditions it uses.

*Your response.*

**Explanation:** How does this tree connect ordinary feature-space partitioning with meta-learning across datasets? What limitation should remain in mind with only twelve tasks?

*Your response.*